In [ ]:
import glob
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from matminer.datasets import load_dataset
from matminer.featurizers.structure import DensityFeatures, GlobalSymmetryFeatures, MaximumPackingEfficiency
from matminer.featurizers.conversions import StructureToComposition
from matminer.featurizers.composition import ElementProperty
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
import os
import numpy as np

In [ ]:
df_full = load_dataset("matbench_mp_gap") 
chunk_size = 1000 # Treating the data in chunks since it keeps crashing
total_rows = len(df_full)

# Initializinf the featurizers
stc = StructureToComposition()
ep = ElementProperty.from_preset(preset_name="magpie")
df_feat = DensityFeatures()
sym_feat = GlobalSymmetryFeatures()
output_filename = "processed_magpie_data.csv"

for i in range(0, total_rows, chunk_size):
    chunk_path = f"chunks/chunk_{i}.csv"
    if os.path.exists(chunk_path):
        print(f"Skipping chunk {i} (already exists)") #this is useful if it crashes halfway
        continue
        
    print(f"Processing rows {i} to {i + chunk_size}...")
    
    df_chunk = df_full.iloc[i : i + chunk_size].copy()
    
    # Featurizing
    df_chunk = stc.featurize_dataframe(df_chunk, "structure", ignore_errors=True)
    df_chunk = ep.featurize_dataframe(df_chunk, col_id="composition", ignore_errors=True)
    df_chunk = df_feat.featurize_dataframe(df_chunk, "structure", ignore_errors=True)
    df_chunk = sym_feat.featurize_dataframe(df_chunk, "structure", ignore_errors=True)
    
    cols_to_keep = [col for col in df_chunk.columns if col not in ['structure', 'composition']]
    df_chunk = df_chunk[cols_to_keep]
    df_chunk.to_csv(chunk_path, index=False)
    
    # Clearing memory
    del df_chunk

print("All chunks featurized and saved to disk.")


# Load all chunks and combine
all_files = glob.glob("chunk_*.csv")
df_final = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
mapping = {"triclinic": 0, "cubic": 1, "orthorhombic": 2, "trigonal": 3,"monoclinic":4,"tetragonal":5,"hexagonal":6}
df_final['crystal_system'] = df_final['crystal_system'].map(mapping)
# Getting my target and features datasets
y = df_final['gap pbe'].values.astype('float32').reshape(-1, 1)
X_df = df_final.drop(columns=['gap pbe'], errors='ignore')

# Filling missing data
imputer = KNNImputer(n_neighbors=5) # I used KNN imputer since it gives a better approximation than just using the usual mean fill
X_imputed = imputer.fit_transform(X_df) 

print(f"Success! Found {X_df.shape[1]} numeric features.")

# Scaling X and putting everything into the GPU
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_tensor = torch.tensor(X_scaled).float().to(device)
y_tensor = torch.tensor(y).float().to(device)

print(f"Final Data Shape: {X_tensor.shape}")
print(f"Target Shape: {y_tensor.shape}")


In [ ]:
#splitting the data into training and testing
total_samples = X_tensor.shape[0]
train_size = int(0.8 * total_samples)

indices = torch.randperm(total_samples)
train_indices = indices[:train_size]
test_indices = indices[train_size:]

X_train, y_train = X_tensor[train_indices], y_tensor[train_indices]
X_test, y_test = X_tensor[test_indices], y_tensor[test_indices]

# Create DataLoaders (this handles the "batches" so the GPU doesn't get overwhelmed)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64)

print(f"Training on: {len(X_train)} samples")
print(f"Testing on: {len(X_test)} samples")

In [ ]:
#Initializing the model,the optimizer and the lossfunction 
from model import BandgapModel
model = BandgapModel(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

In [ ]:
#Training Loop
epochs = 200
model.train()

start_time = time.time()
print("Training started...")

for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()

print(f"Training finished in {time.time() - start_time:.2f} seconds.")

In [ ]:
#Testing 
model.eval()
total_error = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        preds = model(batch_X)
        # Calculate Absolute Error
        error = torch.abs(preds - batch_y)
        total_error += error.sum().item()

mae = total_error / len(X_test)

print("-" * 30)
print(f"MEAN ABSOLUTE ERROR: {mae:.4f} eV")
print("-" * 30)

In [ ]:
#Visualization of the results
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    all_preds = model(X_test).cpu().numpy()
    all_actual = y_test.cpu().numpy()
    
print("\nSample Comparisons:")
for i in range(5):
    print(f"Predicted: {all_preds[i][0]:.2f} eV | Actual: {all_actual[i][0]:.2f} eV")

plt.figure(figsize=(8, 6))
plt.scatter(all_actual, all_preds, alpha=0.3, color='blue')
plt.plot([all_actual.min(), all_actual.max()], [all_actual.min(), all_actual.max()], 'r--', lw=2)
plt.xlabel("Actual Bandgap (eV)")
plt.ylabel("Predicted Bandgap (eV)")
plt.title("Model Accuracy: Predicted vs. Actual")
plt.grid(True)
plt.show()